In [0]:
atm_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/testdatabricks21/schema_bankdata/testvolume1/atm_transactions.csv")

display(atm_df)

In [0]:
branch_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/testdatabricks21/schema_bankdata/testvolume1/branch_transactions.csv")

display(atm_df)

In [0]:
online_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/testdatabricks21/schema_bankdata/testvolume1/online_transactions.csv")

display(atm_df)

After data loading, Add Source Channel

In [0]:
from pyspark.sql.functions import lit

atm_df = atm_df.withColumn(
    "Channel",
    lit("ATM")
)

online_df = online_df.withColumn(
    "Channel",
    lit("ONLINE")
)

branch_df = branch_df.withColumn(
    "Channel",
    lit("BRANCH")
)

Standardize Schema

In [0]:
atm_std = atm_df.select(
    "TransactionID",
    "CustomerID",
    "Amount",
    "TransactionDate",
    "Channel"
)

In [0]:
online_std = online_df.select(
    "TransactionID",
    "CustomerID",
    "Amount",
    "TransactionDate",
    "Channel"
)

In [0]:
branch_std = branch_df.select(
    "TransactionID",
    "CustomerID",
    "Amount",
    "TransactionDate",
    "Channel"
)

Combine All Transactions

In [0]:
unified_df = (
    atm_std
    .union(online_std)
    .union(branch_std)
)

display(unified_df)

Analysis 1: High Value Transactions

In [0]:
high_value_df = unified_df.filter(
    unified_df.Amount > 50000
)

display(high_value_df)

Analysis 2: Total Transactions by Customer

In [0]:
from pyspark.sql.functions import sum

customer_summary = (
    unified_df
    .groupBy("CustomerID")
    .agg(
        sum("Amount").alias("TotalAmount")
    )
)

display(customer_summary)

Analysis 3: Channel Usage

In [0]:
from pyspark.sql.functions import count

channel_summary = (
    unified_df
    .groupBy("Channel")
    .agg(
        count("*").alias("Transactions")
    )
)

display(channel_summary)

Analysis 4: Transactions By Day

In [0]:
daily_summary = (
    unified_df
    .groupBy("TransactionDate")
    .agg(sum("Amount").alias("DayWiseTotalAmt"))
)

display(daily_summary)

Top 5 Customers

Analysis 5: Top 5 Customers

In [0]:
from pyspark.sql.functions import desc

top_customers = (
    customer_summary
    .orderBy(
        desc("TotalAmount")
    )
)

display(top_customers)

Branch vs ATM vs Online Contribution

In [0]:
from pyspark.sql.functions import sum

channel_amount = (
    unified_df
    .groupBy("Channel")
    .agg(sum("Amount").alias("ChannelAmount"))
)

display(channel_amount)

Calculate Grand Total

In [0]:
grand_total = unified_df.agg(
    sum("Amount").alias("TotalAmount")
).collect()[0]["TotalAmount"]

print(grand_total)

Calculate Percentage Contribution

In [0]:
from pyspark.sql.functions import round

channel_percentage = (
    channel_amount
    .withColumn(
        "ContributionPct",
        round(
            (col('ChannelAmount') / grand_total) * 100,
            2
        )
    )
)

display(channel_percentage)

More Spark-Optimized Way (No collect())

In [0]:
from pyspark.sql.functions import sum, round

channel_amount = (
    unified_df
    .groupBy("Channel")
    .agg(sum("Amount").alias("ChannelAmount"))
)

total_df = (
    unified_df
    .agg(sum("Amount").alias("GrandTotal"))
)

channel_percentage = (
    channel_amount
    .crossJoin(total_df)
    .withColumn(
        "ContributionPct",
        round(
            (channel_amount.ChannelAmount / total_df.GrandTotal) * 100,
            2
        )
    )
)

display(channel_percentage)

Databricks visualization. Run in Databricks to view.

Create Catalog and Schema

First check what you have:

In [0]:
spark.sql("SHOW CATALOGS").show()

Create a schema for Gold:

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS testdatabricks21.gold")

Create Gold Tables from DataFrames

In [0]:
customer_summary.write \
.mode("overwrite") \
.saveAsTable("testdatabricks21.gold.customer_summary")

In [0]:
high_value_df.write \
.mode("overwrite") \
.saveAsTable("testdatabricks21.gold.high_value_transactions")

In [0]:
channel_percentage.write \
.mode("overwrite") \
.saveAsTable("testdatabricks21.gold.channel_performance")

In [0]:
daily_summary.write \
.mode("overwrite") \
.saveAsTable("testdatabricks21.gold.daily_summary")

Verify Tables

In [0]:
spark.sql("""
SHOW TABLES IN testdatabricks21.gold
""").show()